# Conversational Chain-of-Thought LLM with Tool Use & MCP Integration

This notebook trains a **conversational chatbot-style LLM from scratch** using a
Chain-of-Thought (COT) dataset, with built-in tool-use and MCP (Model Context Protocol)
server integration.

## Architecture Highlights

| Component | Detail |
|---|---|
| **Normalization** | RMSNorm (LLaMA-style) |
| **Positional Encoding** | Rotary Positional Embeddings (RoPE) |
| **Feed-Forward** | SwiGLU activation |
| **Attention** | Multi-Head Self-Attention with causal masking & KV-Cache |
| **Training Data** | `kaist-ai/CoT-Collection` from HuggingFace |
| **Special Tokens** | `<|system|>`, `<|user|>`, `<|assistant|>`, `<|thinking|>`, `<|/thinking|>`, `<|tool_call|>`, `<|tool_result|>`, `<|end|>` |
| **Training** | Mixed-precision AMP, gradient clipping, cosine LR with warmup |
| **Inference** | Temperature / top-k / top-p sampling with KV-cache |
| **Tool Use** | Built-in calculator, datetime, echo; extensible via MCP servers |

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install -q tiktoken datasets wandb mcp

# ── Imports ───────────────────────────────────────────────────────────
import os
import math
import json
import random
import time
import datetime
import warnings
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import tiktoken
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ── Device ────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────

@dataclass
class ChatLLMConfig:
    """Full configuration for the conversational COT LLM."""

    # ── Model architecture ────────────────────────────────────────────
    base_vocab_size: int = 50257          # GPT-2 tiktoken vocab
    num_special_tokens: int = 8           # special conversational tokens
    vocab_size: int = 50265               # base + special
    embed_dim: int = 384
    num_heads: int = 6
    num_layers: int = 6
    ffn_dim_multiplier: float = 4.0
    max_seq_len: int = 512
    dropout: float = 0.1

    # ── Training ──────────────────────────────────────────────────────
    batch_size: int = 32
    learning_rate: float = 3e-4
    max_steps: int = 5000
    warmup_steps: int = 200
    eval_interval: int = 250
    weight_decay: float = 0.1
    grad_clip: float = 1.0

    # ── Generation ────────────────────────────────────────────────────
    max_gen_len: int = 300
    temperature: float = 0.7
    top_k: int = 50
    top_p: float = 0.9

    # ── Tool / MCP ────────────────────────────────────────────────────
    mcp_server_url: Optional[str] = None
    available_tools: List[Dict[str, str]] = field(default_factory=lambda: [
        {"name": "calculator", "description": "Evaluate a math expression safely."},
        {"name": "web_search", "description": "Search the web for information."},
        {"name": "datetime", "description": "Get current date/time information."},
    ])


config = ChatLLMConfig()
print("ChatLLMConfig created")
print(f"  vocab_size     = {config.vocab_size}")
print(f"  embed_dim      = {config.embed_dim}")
print(f"  num_heads      = {config.num_heads}")
print(f"  num_layers     = {config.num_layers}")
print(f"  max_seq_len    = {config.max_seq_len}")
print(f"  max_steps      = {config.max_steps}")
print(f"  available_tools= {[t['name'] for t in config.available_tools]}")

In [ ]:
# ── Chat Tokenizer ────────────────────────────────────────────────────

class ChatTokenizer:
    """Wraps tiktoken GPT-2 encoding with special conversational tokens."""

    SPECIAL_TOKENS = [
        "<|system|>",
        "<|user|>",
        "<|assistant|>",
        "<|thinking|>",
        "<|/thinking|>",
        "<|tool_call|>",
        "<|tool_result|>",
        "<|end|>",
    ]

    def __init__(self, base_vocab_size: int = 50257):
        self.base_enc = tiktoken.get_encoding("gpt2")
        self.base_vocab_size = base_vocab_size

        # Map special tokens to IDs beyond the base vocab
        self.special_token_to_id: Dict[str, int] = {}
        self.id_to_special_token: Dict[int, str] = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = base_vocab_size + i
            self.special_token_to_id[tok] = tid
            self.id_to_special_token[tid] = tok

        self.vocab_size = base_vocab_size + len(self.SPECIAL_TOKENS)
        self.pad_id = self.special_token_to_id["<|end|>"]
        self.eos_id = self.special_token_to_id["<|end|>"]

    # ── Core encode / decode ──────────────────────────────────────────
    def encode(self, text: str) -> List[int]:
        """Encode text, handling special tokens embedded in the string."""
        if not text:
            return []
        tokens: List[int] = []
        remaining = text
        while remaining:
            # Find the earliest special token in the remaining text
            earliest_pos = len(remaining)
            earliest_tok = None
            for sp in self.SPECIAL_TOKENS:
                pos = remaining.find(sp)
                if pos != -1 and pos < earliest_pos:
                    earliest_pos = pos
                    earliest_tok = sp
            if earliest_tok is None:
                # No more special tokens – encode the rest
                tokens.extend(self.base_enc.encode(remaining, allowed_special=set()))
                break
            # Encode text before the special token
            if earliest_pos > 0:
                tokens.extend(self.base_enc.encode(remaining[:earliest_pos], allowed_special=set()))
            tokens.append(self.special_token_to_id[earliest_tok])
            remaining = remaining[earliest_pos + len(earliest_tok):]
        return tokens

    def decode(self, token_ids: List[int]) -> str:
        """Decode token IDs back to text."""
        if not token_ids:
            return ""
        parts: List[str] = []
        buf: List[int] = []
        for tid in token_ids:
            if tid in self.id_to_special_token:
                if buf:
                    parts.append(self.base_enc.decode(buf))
                    buf = []
                parts.append(self.id_to_special_token[tid])
            else:
                buf.append(tid)
        if buf:
            parts.append(self.base_enc.decode(buf))
        return "".join(parts)

    # ── Chat helpers ──────────────────────────────────────────────────
    def encode_chat_message(self, role: str, content: str) -> List[int]:
        """Encode a single chat message with role tokens."""
        role_token = f"<|{role}|>"
        if role_token not in self.special_token_to_id:
            # Unknown role – fall back to plain encoding
            return self.encode(content)
        return [self.special_token_to_id[role_token]] + self.encode(content) + [self.eos_id]

    def format_conversation(self, messages: List[Dict[str, str]]) -> List[int]:
        """Encode a full conversation: list of {role, content} dicts."""
        tokens: List[int] = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            tokens.extend(self.encode_chat_message(role, content))
        return tokens


tokenizer = ChatTokenizer(config.base_vocab_size)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_token_to_id}")

# Quick sanity check
demo = "<|user|>What is 2+2?<|end|><|assistant|>4<|end|>"
enc = tokenizer.encode(demo)
dec = tokenizer.decode(enc)
print(f"\nEncode/decode test:")
print(f"  Original : {demo}")
print(f"  Token IDs: {enc[:15]}...")
print(f"  Decoded  : {dec}")
assert dec == demo, "Round-trip failed!"
print("  ✓ Round-trip OK")

In [ ]:
# ── COT Dataset ───────────────────────────────────────────────────────

class COTDataset(Dataset):
    """
    Loads kaist-ai/CoT-Collection and formats into conversational COT sequences.

    Each sample becomes:
        <|user|> {source} <|end|>
        <|thinking|> {rationale} <|/thinking|>
        <|assistant|> {target} <|end|>
    """

    def __init__(self, tokenizer: ChatTokenizer, max_len: int = 512,
                 split: str = "train", max_samples: Optional[int] = 50000):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.samples: List[List[int]] = []

        try:
            from datasets import load_dataset
            print(f"Loading kaist-ai/CoT-Collection ({split}) …")
            ds = load_dataset("kaist-ai/CoT-Collection", split=split, streaming=True)
        except Exception as e:
            print(f"⚠ Could not load dataset: {e}")
            print("  → Generating synthetic placeholder data instead.")
            self._generate_synthetic(200)
            return

        count = 0
        for sample in ds:
            if max_samples is not None and count >= max_samples:
                break
            ids = self._format_sample(sample)
            if ids is not None and len(ids) > 4:
                self.samples.append(ids)
                count += 1
            if count % 10000 == 0 and count > 0:
                print(f"  Loaded {count} samples …")

        print(f"Dataset ready: {len(self.samples)} samples")

    # ── Format a single sample ────────────────────────────────────────
    def _format_sample(self, sample: Dict[str, Any]) -> Optional[List[int]]:
        source = sample.get("source", "")
        rationale = sample.get("rationale", "")
        target = sample.get("target", "")

        if not source or not target:
            return None

        text_parts = [f"<|user|>{source.strip()}<|end|>"]
        if rationale and rationale.strip():
            text_parts.append(f"<|thinking|>{rationale.strip()}<|/thinking|>")
        text_parts.append(f"<|assistant|>{target.strip()}<|end|>")

        text = "".join(text_parts)
        try:
            ids = self.tokenizer.encode(text)
        except Exception:
            return None

        # Clamp to max_len
        if len(ids) > self.max_len:
            ids = ids[: self.max_len]
        return ids

    # ── Synthetic fallback ────────────────────────────────────────────
    def _generate_synthetic(self, n: int):
        templates = [
            ("What is {a}+{b}?", "Let me add {a} and {b}.", "{c}"),
            ("Translate 'hello' to French.", "The French word for hello is bonjour.", "bonjour"),
            ("Is {a} even or odd?", "{a} divided by 2 gives remainder {r}.", "{parity}"),
        ]
        for _ in range(n):
            a, b = random.randint(1, 100), random.randint(1, 100)
            c = a + b
            r = a % 2
            parity = "even" if r == 0 else "odd"
            tmpl = random.choice(templates)
            src = tmpl[0].format(a=a, b=b, c=c, r=r, parity=parity)
            rat = tmpl[1].format(a=a, b=b, c=c, r=r, parity=parity)
            tgt = tmpl[2].format(a=a, b=b, c=c, r=r, parity=parity)
            sample = {"source": src, "rationale": rat, "target": tgt}
            ids = self._format_sample(sample)
            if ids is not None:
                self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return torch.tensor(self.samples[idx], dtype=torch.long)


def collate_fn(batch: List[torch.Tensor], pad_id: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """Pad sequences to same length in a batch; return (input, target) pairs."""
    max_len = max(t.size(0) for t in batch)
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    target_ids = torch.full((len(batch), max_len), -100, dtype=torch.long)  # -100 = ignore
    for i, t in enumerate(batch):
        seq_len = t.size(0)
        input_ids[i, :seq_len] = t
        target_ids[i, : seq_len - 1] = t[1:]
        # last position has no target; already -100
    return input_ids, target_ids


print("COTDataset and collate_fn defined.")

In [ ]:
# ── Model Architecture ────────────────────────────────────────────────

# ── RMSNorm ───────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


# ── Rotary Positional Embeddings ──────────────────────────────────────
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if seq_len > self.cos_cached.size(0):
            self._build_cache(seq_len)
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def _rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                     position_offset: int = 0) -> torch.Tensor:
    seq_len = x.size(-2)
    cos = cos[position_offset: position_offset + seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[position_offset: position_offset + seq_len].unsqueeze(0).unsqueeze(0)
    return x * cos + _rotate_half(x) * sin


# ── SwiGLU FFN ────────────────────────────────────────────────────────
class SwiGLU(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


# ── Multi-Head Self-Attention ─────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.0,
                 max_seq_len: int = 2048):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x: torch.Tensor,
                kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                use_cache: bool = False) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # RoPE
        offset = 0
        if kv_cache is not None:
            offset = kv_cache[0].size(2)
        cos, sin = self.rope(offset + T)
        q = apply_rotary_emb(q, cos, sin, position_offset=offset)
        k = apply_rotary_emb(k, cos, sin, position_offset=offset)

        # KV-cache
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v) if use_cache else None

        # Scaled dot-product attention with causal mask
        scale = math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1)) / scale

        # Causal mask
        kv_len = k.size(2)
        causal_mask = torch.triu(torch.ones(T, kv_len, device=x.device, dtype=torch.bool),
                                 diagonal=kv_len - T + 1)
        attn = attn.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.wo(out))
        return out, new_cache


# ── Transformer Block ────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, ffn_multiplier: float,
                 dropout: float, max_seq_len: int):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, dropout, max_seq_len)
        self.norm2 = RMSNorm(dim)
        hidden_dim = int(dim * ffn_multiplier * 2 / 3)
        # Round to nearest multiple of 8 for efficiency
        hidden_dim = ((hidden_dim + 7) // 8) * 8
        self.ffn = SwiGLU(dim, hidden_dim, dropout)

    def forward(self, x: torch.Tensor,
                kv_cache=None, use_cache: bool = False):
        h, new_cache = self.attn(self.norm1(x), kv_cache=kv_cache, use_cache=use_cache)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, new_cache


# ── Main Model ────────────────────────────────────────────────────────
class ConversationalLLM(nn.Module):
    def __init__(self, cfg: ChatLLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.embed_dim)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg.embed_dim, cfg.num_heads, cfg.ffn_dim_multiplier,
                             cfg.dropout, cfg.max_seq_len)
            for _ in range(cfg.num_layers)
        ])
        self.norm = RMSNorm(cfg.embed_dim)
        self.head = nn.Linear(cfg.embed_dim, cfg.vocab_size, bias=False)

        # Weight tying
        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"ConversationalLLM: {n_params / 1e6:.2f}M parameters")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor,
                kv_caches: Optional[List] = None,
                use_cache: bool = False) -> Tuple[torch.Tensor, Optional[List]]:
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx))
        new_caches = []
        for i, block in enumerate(self.blocks):
            cache_i = kv_caches[i] if kv_caches is not None else None
            x, nc = block(x, kv_cache=cache_i, use_cache=use_cache)
            new_caches.append(nc)
        x = self.norm(x)
        logits = self.head(x)
        return logits, new_caches if use_cache else None

    # ── Generation ────────────────────────────────────────────────────
    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 300,
                 temperature: float = 0.7, top_k: int = 50, top_p: float = 0.9,
                 eos_id: Optional[int] = None) -> torch.Tensor:
        """Auto-regressive generation with temperature, top-k, top-p sampling."""
        # Bounds checking
        temperature = max(temperature, 1e-4)
        top_k = max(top_k, 1)
        top_p = min(max(top_p, 0.0), 1.0)
        max_new_tokens = min(max(max_new_tokens, 1), self.cfg.max_seq_len)

        self.eval()
        generated = input_ids.clone()
        kv_caches = None

        for _ in range(max_new_tokens):
            # Clamp sequence length
            if generated.size(1) > self.cfg.max_seq_len:
                generated = generated[:, -self.cfg.max_seq_len:]
                kv_caches = None  # reset cache

            if kv_caches is not None:
                inp = generated[:, -1:]
            else:
                inp = generated

            logits, kv_caches = self.forward(inp, kv_caches=kv_caches, use_cache=True)
            logits = logits[:, -1, :] / temperature

            # Top-k filtering
            if top_k > 0 and top_k < logits.size(-1):
                topk_vals, _ = torch.topk(logits, top_k, dim=-1)
                logits[logits < topk_vals[:, -1:]] = float("-inf")

            # Top-p (nucleus) filtering
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                mask = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[mask] = float("-inf")
                logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_tok], dim=1)

            if eos_id is not None and (next_tok == eos_id).all():
                break

        return generated

    # ── Tool-call detection ───────────────────────────────────────────
    @staticmethod
    def parse_tool_calls(text: str) -> List[Dict[str, Any]]:
        """Parse <|tool_call|>...<|end|> blocks from generated text."""
        calls = []
        marker = "<|tool_call|>"
        end_marker = "<|end|>"
        idx = 0
        while True:
            start = text.find(marker, idx)
            if start == -1:
                break
            end = text.find(end_marker, start + len(marker))
            if end == -1:
                break
            payload = text[start + len(marker): end].strip()
            try:
                call = json.loads(payload)
                calls.append(call)
            except json.JSONDecodeError:
                # Best-effort: store raw text
                calls.append({"raw": payload, "error": "invalid JSON"})
            idx = end + len(end_marker)
        return calls


print("Model architecture defined.")

In [ ]:
# ── Training Utilities ─────────────────────────────────────────────────

def get_lr(step: int, warmup_steps: int, max_steps: int, max_lr: float,
           min_lr: float = 1e-6) -> float:
    """Cosine learning rate schedule with linear warmup."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def train_model(model: ConversationalLLM, train_loader: DataLoader,
                val_loader: Optional[DataLoader], cfg: ChatLLMConfig,
                device: torch.device) -> Dict[str, List[float]]:
    """Training loop with AMP, gradient clipping, NaN detection, checkpointing."""

    model.to(device)
    model.train()

    # Separate weight-decay groups
    decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW([
        {"params": decay_params, "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=cfg.learning_rate, betas=(0.9, 0.95))

    scaler = GradScaler("cuda") if device.type == "cuda" else None
    history: Dict[str, List[float]] = {"train_loss": [], "val_loss": [], "lr": [], "step": []}

    best_val_loss = float("inf")
    data_iter = iter(train_loader)
    nan_skip_count = 0

    print(f"Starting training for {cfg.max_steps} steps …")
    t0 = time.time()

    for step in range(cfg.max_steps):
        # Fetch batch (cycle through data)
        try:
            input_ids, target_ids = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            input_ids, target_ids = next(data_iter)

        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        # LR schedule
        lr = get_lr(step, cfg.warmup_steps, cfg.max_steps, cfg.learning_rate)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        # Forward + backward with AMP
        optimizer.zero_grad(set_to_none=True)
        if device.type == "cuda":
            with autocast("cuda"):
                logits, _ = model(input_ids)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                       target_ids.view(-1), ignore_index=-100)
            # NaN / Inf detection
            if torch.isnan(loss) or torch.isinf(loss):
                nan_skip_count += 1
                if nan_skip_count % 50 == 1:
                    print(f"  ⚠ NaN/Inf loss at step {step} (skipped {nan_skip_count} total)")
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, _ = model(input_ids)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   target_ids.view(-1), ignore_index=-100)
            if torch.isnan(loss) or torch.isinf(loss):
                nan_skip_count += 1
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

        history["train_loss"].append(loss.item())
        history["lr"].append(lr)
        history["step"].append(step)

        # Logging
        if step % 100 == 0:
            elapsed = time.time() - t0
            print(f"  step {step:>5d}/{cfg.max_steps} | loss {loss.item():.4f} | "
                  f"lr {lr:.2e} | {elapsed:.1f}s")

        # Evaluation
        if val_loader is not None and (step + 1) % cfg.eval_interval == 0:
            val_loss = evaluate(model, val_loader, device)
            history["val_loss"].append(val_loss)
            print(f"  ── val_loss {val_loss:.4f} at step {step + 1}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), "best_cot_llm.pt")
                print(f"       ✓ New best model saved (val_loss={val_loss:.4f})")
            model.train()

    total_time = time.time() - t0
    print(f"\nTraining complete in {total_time:.1f}s")
    print(f"  NaN/Inf skips: {nan_skip_count}")
    if best_val_loss < float("inf"):
        print(f"  Best val loss: {best_val_loss:.4f}")
    return history


@torch.no_grad()
def evaluate(model: ConversationalLLM, val_loader: DataLoader,
             device: torch.device, max_batches: int = 50) -> float:
    model.eval()
    total_loss = 0.0
    n = 0
    for i, (input_ids, target_ids) in enumerate(val_loader):
        if i >= max_batches:
            break
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        logits, _ = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               target_ids.view(-1), ignore_index=-100)
        if not (torch.isnan(loss) or torch.isinf(loss)):
            total_loss += loss.item()
            n += 1
    return total_loss / max(n, 1)


print("Training functions defined.")

In [ ]:
# ── Execute Training ──────────────────────────────────────────────────

try:
    # Config & tokenizer already created above
    print("Preparing data …")
    dataset = COTDataset(tokenizer, max_len=config.max_seq_len, split="train",
                         max_samples=50000)

    # Train / val split
    val_size = max(1, int(len(dataset) * 0.05))
    train_size = len(dataset) - val_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    pad_id = tokenizer.pad_id
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              collate_fn=lambda b: collate_fn(b, pad_id), drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                            collate_fn=lambda b: collate_fn(b, pad_id))

    print(f"Train: {train_size}  Val: {val_size}  Batches/epoch: {len(train_loader)}")

    # Model
    model = ConversationalLLM(config)
    print(f"Model on {device}")

    # Train
    history = train_model(model, train_loader, val_loader, config, device)

except Exception as e:
    print(f"Training error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────

try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Training loss (smoothed)
    if history.get("train_loss"):
        raw = history["train_loss"]
        window = min(50, len(raw) // 4) or 1
        smoothed = np.convolve(raw, np.ones(window) / window, mode="valid")
        axes[0].plot(smoothed, linewidth=0.8)
        axes[0].set_title("Training Loss (smoothed)")
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].grid(True, alpha=0.3)

    # Validation loss
    if history.get("val_loss"):
        val_steps = list(range(config.eval_interval,
                               config.eval_interval * (len(history["val_loss"]) + 1),
                               config.eval_interval))
        axes[1].plot(val_steps, history["val_loss"], "o-", color="orange")
        axes[1].set_title("Validation Loss")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Loss")
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()
    print("Training curves saved to training_curves.png")
except Exception as e:
    print(f"Visualization error: {e}")

In [ ]:
# ── Generation Utilities & Testing ────────────────────────────────────

def generate_response(model: ConversationalLLM, tokenizer: ChatTokenizer,
                      prompt: str, max_new_tokens: int = 300,
                      temperature: float = 0.7, top_k: int = 50,
                      top_p: float = 0.9, device: torch.device = device) -> str:
    """Generate a response with full edge-case handling."""
    # Edge cases
    if not prompt or not prompt.strip():
        return "[Empty prompt — nothing to generate]"
    temperature = max(min(temperature, 2.0), 1e-4)
    top_k = max(top_k, 1)
    top_p = max(min(top_p, 1.0), 0.0)
    max_new_tokens = max(min(max_new_tokens, model.cfg.max_seq_len), 1)

    # Tokenize
    try:
        ids = tokenizer.encode(prompt)
    except Exception:
        ids = tokenizer.encode("Hello")
    if not ids:
        return "[Tokenization produced no tokens]"

    # Truncate to fit
    if len(ids) > model.cfg.max_seq_len - 10:
        ids = ids[-(model.cfg.max_seq_len - 10):]

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens,
                                temperature=temperature, top_k=top_k, top_p=top_p,
                                eos_id=tokenizer.eos_id)
    generated = output_ids[0, len(ids):].tolist()
    return tokenizer.decode(generated)


# Load best checkpoint if available
ckpt_path = "best_cot_llm.pt"
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print(f"Loaded best model from {ckpt_path}")
else:
    print("No best checkpoint found; using final model weights.")

model.to(device)
model.eval()

# ── Test prompts ──────────────────────────────────────────────────────
test_prompts = [
    "<|user|>What is 15 multiplied by 7?<|end|><|assistant|>",
    "<|user|>Explain photosynthesis briefly.<|end|><|assistant|>",
    "<|user|>Translate 'good morning' to Spanish.<|end|><|assistant|>",
]

print("=" * 60)
print("Sample Generations")
print("=" * 60)
for p in test_prompts:
    resp = generate_response(model, tokenizer, p, max_new_tokens=100, temperature=0.8)
    print(f"\nPrompt : {p}")
    print(f"Response: {resp[:200]}")
    print("-" * 60)

In [ ]:
# ── Tool Executor & MCP Client ────────────────────────────────────────

import ast
import operator

class ToolExecutor:
    """Built-in tool executor with safe implementations."""

    SAFE_OPS = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg,
    }

    def execute(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        handlers = {
            "calculator": self._calculator,
            "datetime": self._datetime,
            "echo": self._echo,
            "web_search": self._web_search,
        }
        handler = handlers.get(tool_name)
        if handler is None:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        try:
            return handler(arguments)
        except Exception as e:
            return json.dumps({"error": str(e)})

    def _calculator(self, args: Dict[str, Any]) -> str:
        expr = str(args.get("expression", ""))
        if not expr:
            return json.dumps({"error": "No expression provided"})
        try:
            result = self._safe_eval(expr)
            return json.dumps({"result": result})
        except Exception as e:
            return json.dumps({"error": f"Eval failed: {e}"})

    def _safe_eval(self, expr: str) -> float:
        """Evaluate arithmetic expression safely using AST."""
        tree = ast.parse(expr, mode="eval")
        return self._eval_node(tree.body)

    def _eval_node(self, node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            op = self.SAFE_OPS.get(type(node.op))
            if op is None:
                raise ValueError(f"Unsupported op: {type(node.op).__name__}")
            return op(self._eval_node(node.left), self._eval_node(node.right))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -self._eval_node(node.operand)
        raise ValueError(f"Unsupported node: {type(node).__name__}")

    def _datetime(self, args: Dict[str, Any]) -> str:
        fmt = args.get("format", "%Y-%m-%d %H:%M:%S")
        return json.dumps({"datetime": datetime.datetime.now().strftime(fmt)})

    def _echo(self, args: Dict[str, Any]) -> str:
        return json.dumps({"echo": args.get("message", "")})

    def _web_search(self, args: Dict[str, Any]) -> str:
        query = args.get("query", "")
        return json.dumps({"note": "Web search not available in this environment",
                           "query": query})


class MCPClient:
    """Minimal MCP (Model Context Protocol) server client."""

    def __init__(self, server_url: Optional[str] = None):
        self.server_url = server_url
        self.connected = False
        if server_url:
            self._connect()

    def _connect(self):
        try:
            # Attempt a lightweight handshake
            import urllib.request
            req = urllib.request.Request(
                self.server_url.rstrip("/") + "/health",
                method="GET"
            )
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    self.connected = True
                    print(f"MCP: Connected to {self.server_url}")
        except Exception as e:
            self.connected = False
            print(f"MCP: Connection to {self.server_url} failed ({e}). "
                  "Falling back to local tools.")

    def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> Optional[str]:
        """Send a tool call to the MCP server. Returns None on failure."""
        if not self.connected or not self.server_url:
            return None
        try:
            import urllib.request
            payload = json.dumps({"tool": tool_name, "arguments": arguments}).encode()
            req = urllib.request.Request(
                self.server_url.rstrip("/") + "/tool/call",
                data=payload,
                headers={"Content-Type": "application/json"},
                method="POST"
            )
            with urllib.request.urlopen(req, timeout=10) as resp:
                return resp.read().decode()
        except Exception as e:
            print(f"MCP tool call failed: {e}")
            return None


def execute_tool_call(call: Dict[str, Any], executor: ToolExecutor,
                      mcp_client: Optional[MCPClient] = None) -> str:
    """Dispatch a tool call to MCP server (if available) or local executor."""
    if "error" in call:
        return json.dumps({"error": call.get("error", "parse error"),
                           "raw": call.get("raw", "")})
    tool_name = call.get("name") or call.get("tool", "")
    arguments = call.get("arguments") or call.get("args", {})

    # Try MCP first
    if mcp_client is not None:
        result = mcp_client.call_tool(tool_name, arguments)
        if result is not None:
            return result

    # Fall back to local
    return executor.execute(tool_name, arguments)


# Instantiate
tool_executor = ToolExecutor()
mcp_client = MCPClient(config.mcp_server_url)  # None by default

# Quick demo
print("Tool executor demo:")
print("  calculator:", tool_executor.execute("calculator", {"expression": "3 * (4 + 5)"}))
print("  datetime  :", tool_executor.execute("datetime", {}))
print("  echo      :", tool_executor.execute("echo", {"message": "hello MCP!"}))

In [ ]:
# ── Interactive Chat Mode ─────────────────────────────────────────────

def interactive_chat(model: ConversationalLLM, tokenizer: ChatTokenizer,
                     executor: ToolExecutor, mcp: Optional[MCPClient] = None,
                     system_prompt: str = "You are a helpful assistant that thinks step by step.",
                     device: torch.device = device):
    """Multi-turn interactive chat with COT display and tool-use support."""

    conversation: List[Dict[str, str]] = [
        {"role": "system", "content": system_prompt}
    ]
    print("=" * 60)
    print("Interactive Chat  (type 'exit' or 'quit' to stop)")
    print("=" * 60)
    print(f"System: {system_prompt}\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Session ended]")
            break

        # Edge cases
        if not user_input:
            print("[Empty input — please type something]\n")
            continue
        if user_input.lower() in ("exit", "quit", "/exit", "/quit"):
            print("[Goodbye!]")
            break
        if user_input.lower() in ("/clear", "/reset"):
            conversation = [{"role": "system", "content": system_prompt}]
            print("[Conversation cleared]\n")
            continue

        # Truncate very long input
        if len(user_input) > 2000:
            user_input = user_input[:2000]
            print("  (input truncated to 2000 chars)")

        conversation.append({"role": "user", "content": user_input})

        # Build prompt
        prompt_tokens = tokenizer.format_conversation(conversation)
        # Reserve room for generation
        max_prompt = model.cfg.max_seq_len - 150
        if len(prompt_tokens) > max_prompt:
            prompt_tokens = prompt_tokens[-max_prompt:]

        prompt_text = tokenizer.decode(prompt_tokens)
        prompt_text += "<|assistant|>"

        # Generate
        response = generate_response(model, tokenizer, prompt_text,
                                     max_new_tokens=config.max_gen_len,
                                     temperature=config.temperature,
                                     top_k=config.top_k, top_p=config.top_p)

        # Display thinking if present
        if "<|thinking|>" in response and "<|/thinking|>" in response:
            think_start = response.index("<|thinking|>") + len("<|thinking|>")
            think_end = response.index("<|/thinking|>")
            thinking = response[think_start:think_end].strip()
            print(f"  💭 Thinking: {thinking}")
            # Remove thinking from display response
            response = response[:response.index("<|thinking|>")] + response[think_end + len("<|/thinking|>"):]

        # Check for tool calls
        tool_calls = ConversationalLLM.parse_tool_calls(response)
        if tool_calls:
            for tc in tool_calls:
                print(f"  🔧 Tool call: {tc}")
                result = execute_tool_call(tc, executor, mcp)
                print(f"  📎 Result: {result}")
                # Append tool result to conversation
                conversation.append({"role": "tool_result", "content": result})

        # Clean up response for display
        clean = response
        for tag in ["<|end|>", "<|assistant|>", "<|tool_call|>", "<|tool_result|>"]:
            clean = clean.replace(tag, "")
        clean = clean.strip()

        print(f"Assistant: {clean}\n")
        conversation.append({"role": "assistant", "content": clean})


# Note: In Colab, run interactive_chat(model, tokenizer, tool_executor, mcp_client)
# Skipping auto-run in notebook to avoid blocking execution.
print("interactive_chat() is ready. Call it to start a chat session:")
print("  interactive_chat(model, tokenizer, tool_executor, mcp_client)")

In [ ]:
# ── Save & Export ─────────────────────────────────────────────────────

save_dir = "cot_llm_artifacts"
os.makedirs(save_dir, exist_ok=True)

# 1) Model weights
model_path = os.path.join(save_dir, "model.pt")
torch.save(model.state_dict(), model_path)
print(f"✓ Model weights saved to {model_path}")

# 2) Config
config_path = os.path.join(save_dir, "config.json")
with open(config_path, "w") as f:
    cfg_dict = asdict(config)
    json.dump(cfg_dict, f, indent=2)
print(f"✓ Config saved to {config_path}")

# 3) Tokenizer info
tok_path = os.path.join(save_dir, "tokenizer_info.json")
with open(tok_path, "w") as f:
    json.dump({
        "base_encoding": "gpt2",
        "base_vocab_size": tokenizer.base_vocab_size,
        "special_tokens": tokenizer.special_token_to_id,
        "total_vocab_size": tokenizer.vocab_size,
    }, f, indent=2)
print(f"✓ Tokenizer info saved to {tok_path}")

# 4) Training log
if "history" in dir():
    log_path = os.path.join(save_dir, "training_log.json")
    # Convert numpy types for JSON serialization
    serializable = {}
    for k, v in history.items():
        serializable[k] = [float(x) for x in v]
    with open(log_path, "w") as f:
        json.dump(serializable, f)
    print(f"✓ Training log saved to {log_path}")

# 5) Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n{'=' * 50}")
print(f"Model Summary")
print(f"{'=' * 50}")
print(f"  Total parameters    : {total_params:,}")
print(f"  Trainable parameters: {trainable:,}")
print(f"  Model size (approx) : {total_params * 4 / 1e6:.1f} MB (FP32)")
print(f"  Architecture        : {config.num_layers}L / {config.num_heads}H / {config.embed_dim}D")
print(f"  Max sequence length : {config.max_seq_len}")
print(f"  Vocab size          : {config.vocab_size}")
print(f"  Artifacts directory : {save_dir}/")
print(f"{'=' * 50}")
print("Done! 🎉")